# 08 - Scenario Modeling And Sensitivity Analysis

This notebook creates an interactive scenario matrix and core sensitivity views for SUMR.

**Model policy**
- Uses explicit, user-visible assumption pins.
- Does not hide defaults for supply/price.
- Keeps calculations deterministic and inspectable.


## Steps

1. Pull latest TVL snapshot (optional).
2. Resolve circulating supply from evidence snapshot.
3. Build scenario matrix via `src.analysis.scenarios.build_scenario_matrix`.
4. Visualize yield sensitivity and sample position outcomes.


In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "notebooks" / "notebook_utils.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError("Could not locate repository root containing notebooks/notebook_utils.py")

if root.as_posix() not in sys.path:
    sys.path.insert(0, root.as_posix())

from notebooks import notebook_utils as nbx
ROOT = nbx.setup_notebook_context()
print(f"Project root: {ROOT}")
print(f"Notebook run started (UTC): {nbx.utc_now_iso()}")


In [ ]:
from src.analysis.scenarios import build_scenario_matrix

REFRESH_DEFILLAMA = True
if REFRESH_DEFILLAMA:
    nbx.refresh_defillama_snapshots()

lazy_tvl_path = nbx.latest_defillama_snapshot("lazy-summer-protocol", "tvl")
lazy_tvl = nbx.protocol_tvl_df(nbx.load_json(lazy_tvl_path))
current_tvl = float(lazy_tvl["totalLiquidityUSD"].iloc[-1])

EVIDENCE_DIR = nbx.latest_evidence_dir()
if EVIDENCE_DIR is None:
    raise FileNotFoundError("No evidence dir found.")

supply_snapshot = nbx.load_json(EVIDENCE_DIR / "sumr_supply_snapshot.json")
circ_supply = float(supply_snapshot.get("totalSupply_tokens") or 0.0)

TOKEN_PRICE_USD = 0.003319

print("current_tvl", current_tvl)
print("circulating_supply", circ_supply)
print("token_price", TOKEN_PRICE_USD)


In [ ]:
scenario_df = build_scenario_matrix(
    current_tvl=current_tvl,
    circ_supply=circ_supply,
    token_price=TOKEN_PRICE_USD,
)

display(scenario_df.head())
print("scenario rows:", len(scenario_df))


In [ ]:
slice_df = scenario_df[
    (scenario_df["tvl_multiplier"] == 1.0)
    & (scenario_df["staking_ratio"] == 0.30)
].copy()

heat = slice_df.pivot(index="fee_rate", columns="staker_share", values="revenue_yield_on_mcap")

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
sns.heatmap(heat * 100.0, annot=True, fmt=".2f", cmap="YlGnBu", cbar_kws={"label": "Yield on mcap (%)"}, ax=ax)
ax.set_title("Yield-On-Mcap Sensitivity (TVL x1.0, Staking ratio 30%)")
ax.set_xlabel("Staker share")
ax.set_ylabel("Fee rate")
plt.show()


In [ ]:
position_tokens = 1_000_000
entry_value = position_tokens * TOKEN_PRICE_USD

case_specs = [
    ("Downside", 0.5, 0.0010, 0.10, 0.30),
    ("Base", 1.0, 0.0066, 0.20, 0.30),
    ("Upside", 2.0, 0.0100, 0.30, 0.30),
]

rows = []
for name, tvl_mult, fee_rate, staker_share, staking_ratio in case_specs:
    row = scenario_df[
        (scenario_df["tvl_multiplier"] == tvl_mult)
        & (scenario_df["fee_rate"] == fee_rate)
        & (scenario_df["staker_share"] == staker_share)
        & (scenario_df["staking_ratio"] == staking_ratio)
    ].iloc[0]

    annual_yield = float(row["revenue_yield_on_staked"] or 0.0)
    annual_cash = entry_value * annual_yield
    three_year_cash = annual_cash * 3

    rows.append(
        {
            "case": name,
            "annual_cash_yield_usd": annual_cash,
            "3y_cash_yield_usd": three_year_cash,
            "yield_on_staked": annual_yield,
        }
    )

case_df = pd.DataFrame(rows)
display(case_df)

fig, ax = plt.subplots(figsize=(8, 4.5), constrained_layout=True)
ax.bar(case_df["case"], case_df["3y_cash_yield_usd"], color=["#d62828", "#457b9d", "#2a9d8f"])
ax.set_title("Illustrative 3-Year Cash Yield (1,000,000 SUMR position)")
ax.set_ylabel("USD")
plt.show()
